# Preparing the dataset

In [ ]:
import pandas as pd

data = './data/Focus.csv'
df = pd.read_csv(data)


df = df.drop(['Track URI', 'Album Name', 'Artist Name(s)',
       'Release Date', 'Popularity', 'Explicit', 'Added By',
       'Added At', 'Record Label',
       'Time Signature', 'Duration (ms)', 'Genres'], axis=1)
print(df.head())

   Danceability  Energy  Key  Loudness  Mode  Speechiness  Acousticness  \
0         0.459   0.171    0   -13.782     1       0.0395         0.860   
1         0.508   0.178    7   -12.266     1       0.0292         0.958   
2         0.629   0.281   11   -15.124     1       0.0374         0.805   
3         0.630   0.228    7   -14.143     1       0.0401         0.921   
4         0.698   0.155   11   -17.919     1       0.0338         0.963   

   Instrumentalness  Liveness  Valence    Tempo  Time Signature  
0             0.280    0.1060   0.1820  179.655               3  
1             0.877    0.1730   0.3740   74.367               4  
2             0.789    0.0893   0.1580   65.634               4  
3             0.940    0.1240   0.0931   89.905               4  
4             0.878    0.0882   0.1950   93.665               4  


In [ ]:

def normalize(column):
    col_max = df[column].max()
    col_min = df[column].min()
    df[column] = (df[column] - col_min) / (col_max - col_min)

normalize('Tempo')


In [ ]:
def key_similarity(key1, mode1, key2, mode2):
    """
    Calculate similarity between two musical keys (0-1 scale, 1 = identical).
    Uses circle of fifths harmonic distance.
    
    Args:
        key1, key2: str - Key name (e.g., 'C', 'F#', 'Bb')
        mode1, mode2: str - 'major' or 'minor'
    
    Returns:
        float: Similarity score (0 = maximally distant, 1 = identical)
    """
    # Circle of fifths positions (0-11)
    circle_positions = {
        'C': 0, 'G': 1, 'D': 2, 'A': 3, 'E': 4, 'B': 5,
        'F#': 6, 'Gb': 6, 'Db': 7, 'C#': 7, 'Ab': 8, 'Eb': 9, 
        'Bb': 10, 'F': 11
    }
    
    # Convert minor keys to their relative major (3 semitones up)
    # On circle of fifths: minor = 3 positions counter-clockwise from relative major
    def to_major_position(key, mode):
        pos = circle_positions.get(key)
        if pos is None:
            raise ValueError(f"Invalid key: {key}")
        
        if mode.lower() == 'minor':
            # A minor -> C major (shift 3 positions clockwise)
            pos = (pos + 3) % 12
        return pos
    
    pos1 = to_major_position(key1, mode1)
    pos2 = to_major_position(key2, mode2)
    
    # Calculate shortest distance around the circle (0-6)
    distance = abs(pos1 - pos2)
    distance = min(distance, 12 - distance)
    
    # Convert to similarity score (0-1)
    # Distance 0 = identical (similarity 1.0)
    # Distance 6 = opposite (similarity 0.0)
    similarity = 1 - (distance / 6)
    
    return similarity

0.7


In [ ]:
features = ['Danceability', 'Energy', 'Speechiness', 'Acousticness', 'Liveness', 'Valence', 'Tempo']

def vector_mod(row):
    s = 0.0
    for col in features:
        s += row[col] ** 2
    return s ** 0.5

def dot_product(row1, row2):
    s = 0.0
    for col in features:
        s += row1[col] * row2[col]
    return s

def cosine_similarity(row1, row2):
    dp = dot_product(row1, row2)
    mod1 = vector_mod(row1)
    mod2 = vector_mod(row2)
    if mod1 == 0 or mod2 == 0:
        return 0.0
    return dp / (mod1 * mod2)


0.9099878588617211


In [ ]:
def Similarity(song1, song2):
    first = df.iloc[song1]
    second = df.iloc[song2]
    key_similarity = key_similarity(first['Key'], second['Key'], first['Mode'], second['Mode'])
    cosine_sim = cosine_similarity(first, second)
    return ( 2* cosine_sim + key_similarity) / 3

In [ ]:

edges = []

for i in range(len(df)):
    for j in range(i+1, len(df)):
        sim = Similarity(i, j)
        edges.append((i, j, sim))
        print(f"Similarity between song {i} and song {j}: {sim:.4f}")


In [ ]:
import networkx as nx

def optimize_playlist_order(G, method='2opt_multistart'):
    """
    Optimized playlist ordering with smooth transitions
    
    Args:
        G: NetworkX graph with similarity weights
        method: 'greedy', 'multistart', '2opt', '2opt_multistart', 'mst', 'mst_2opt'
    """
    if method == '2opt_multistart':
        # Best approach: multiple starts + local optimization
        path, score = find_best_path_multistart(G, num_starts=20)
        path = two_opt_improve(G, path)
        return path
    # ... other methods

def find_best_path_multistart(G, num_starts=None):
    """Try multiple starting points, pick best total similarity"""
    all_nodes = list(G.nodes())
    if num_starts is None:
        num_starts = min(10, len(all_nodes))
    
    best_path = None
    best_score = -float('inf')
    
    import random
    start_nodes = random.sample(all_nodes, num_starts) if len(all_nodes) > num_starts else all_nodes
    
    for start_node in start_nodes:
        path = greedy_path(G, start_node, all_nodes)
        score = calculate_path_score(G, path)
        
        if score > best_score:
            best_score = score
            best_path = path
    
    return best_path, best_score

def greedy_path(G, start, all_nodes):
    """Build path greedily from start node"""
    path = [start]
    visited = {start}
    current = start
    
    while len(visited) < len(all_nodes):
        neighbors = [(n, G[current][n]['weight']) for n in all_nodes 
                     if n not in visited]
        if not neighbors:
            break
        next_node = max(neighbors, key=lambda x: x[1])[0]
        path.append(next_node)
        visited.add(next_node)
        current = next_node
    
    return path

def two_opt_improve(G, path, max_iterations=1000):
    """Iteratively improve path by reversing segments"""
    improved = True
    iteration = 0
    
    while improved and iteration < max_iterations:
        improved = False
        iteration += 1
        
        for i in range(1, len(path) - 2):
            for j in range(i + 1, len(path)):
                if j - i == 1:
                    continue
                
                # Calculate gain from reversing path[i:j]
                current_edges = (
                    G[path[i-1]][path[i]]['weight'] + 
                    G[path[j-1]][path[j]]['weight'] if j < len(path) else 0
                )
                new_edges = (
                    G[path[i-1]][path[j-1]]['weight'] + 
                    G[path[i]][path[j]]['weight'] if j < len(path) else 0
                )
                
                if new_edges > current_edges:
                    path[i:j] = reversed(path[i:j])
                    improved = True
    
    return path

def calculate_path_score(G, path):
    """Calculate total similarity across all transitions"""
    return sum(G[path[i]][path[i+1]]['weight'] for i in range(len(path)-1))
